# Multi-channel

In [2]:
#Libraries
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, correlate
from scipy.ndimage import uniform_filter1d

path = r"C:\spike-denoising-synthetic\data\real" 
sample_rate = 30000

In [9]:
# Load all good unit channels [ch3, ch9, ch10, ch14, ch19, ch25, ch30] 
ttls = np.fromfile(path + r"\TTLs_2.bin", dtype='<f8')
t_start = ttls[900]
start_idx = int(t_start * sample_rate)
n_samples = int(300 * sample_rate)
t_end = t_start + n_samples / sample_rate

In [10]:
def load_ch(n):
    f = path + rf"\100_RhythmData_CH{n}_2_filtered_blanked.bin"
    return np.fromfile(f, dtype='<f8', count=n_samples, offset=start_idx * 8)
good_channels = [3, 9, 10, 14, 19, 25, 30]

gt = h5py.File(path + r"\grouped_unit_1.mat", "r")
sigs, marks = {}, {}
for n in good_channels:
    sigs[n] = load_ch(n)
    zc = gt[f"CH{n}"]["aligned_zc_locs"][:].flatten()
    marks[n] = zc[(zc >= t_start) & (zc <= t_end)]
    print(f"CH{n}: {len(marks[n])} spikes, MAD {np.median(np.abs(sigs[n]))/0.6745:.2f}")

CH3: 21 spikes, MAD 5.00
CH9: 63 spikes, MAD 4.78
CH10: 14 spikes, MAD 4.85
CH14: 72 spikes, MAD 4.86
CH19: 71 spikes, MAD 4.94
CH25: 74 spikes, MAD 4.60
CH30: 72 spikes, MAD 4.77


In [8]:
# Measure each channel's delay relatie to ch 19.
ref = marks[19]
print("propagation delays relative to CH19:")
for n in good_channels:
    if n == 19:
        print(f"CH{n}: reference")
        continue
    offsets = []
    for ch19_spike_time in ref:
        time_diffs = marks[n] - ch19_spike_time
        near = time_diffs[np.abs(time_diffs) < 0.003]   # same spike on both channels
        if len(near):
            offsets.append(near[np.argmin(np.abs(near))])
    offsets = np.array(offsets) * 1000
    if len(offsets):
        print(f"CH{n}: median {np.median(offsets):+.2f} ms (matched {len(offsets)}, std {offsets.std():.2f})")
    else:
        print(f"CH{n}: no matches")

propagation delays relative to CH19:
CH3: median -0.03 ms (matched 21, std 0.03)
CH9: median +0.23 ms (matched 59, std 0.02)
CH10: median +0.40 ms (matched 14, std 0.02)
CH14: median +0.10 ms (matched 68, std 0.02)
CH19: reference
CH25: median +0.33 ms (matched 70, std 0.02)
CH30: median +0.17 ms (matched 68, std 0.03)
